<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 25. One model vs one model per instrument, events vs every bar

**Question from the meeting:** is it better to train one pooled model on all instruments or one model per instrument, and does it matter whether we train on pattern events only or on every bar?

**Design (2 x 2)**
- Pooled: one model trained on all 24 tickers before the cutoff. Per-instrument: one model per ticker, trained only on that ticker before the cutoff.
- Events: only bars where a pattern detector fires. Bars: every bar is labelled by the triple-barrier rule, with recency features.
- Same forward split for everything: train before 2021-12-15 (purged), test from 2022-01-01. Same model settings as before (Random Forest, 300 trees, depth 8, balanced). Metric: F1 macro per ticker, then averaged over tickers.

**Caution when comparing rows:** events and bars have different label definitions and different test samples. The fairest comparison is the "bars, evaluated on event bars only" row against the events row.

In [ ]:
import sys, os
for m in list(sys.modules.keys()):
    if m.startswith('src'):
        del sys.modules[m]

REPO_DIR  = '/content/regime-aware-ml-trading'
PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
if not os.path.isdir(PROJ_ROOT):
    os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
else:
    os.system(f'cd {REPO_DIR} && git pull -q')
os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
print("src exists:", os.path.isdir(os.path.join(PROJ_ROOT, 'src')))

In [ ]:
from src.experiments.helpers import build_event_data, build_bar_data

ALL = ['SPY','QQQ','NVDA','GOOGL','AAPL','MSFT','AMZN','META','AVGO','TSLA','COST','ADBE',
       'AMD','NFLX','INTC','CSCO','JNJ','KO','WMT','PG','CAT','F','XOM','JPM']

event_data, bar_data = {}, {}
for tk in ALL:
    try:
        event_data[tk] = build_event_data(tk)
        bar_data[tk] = build_bar_data(tk, raw=event_data[tk]['raw'])
        print(f"{tk}: {len(event_data[tk]['labels'])} events, {len(bar_data[tk]['labels'])} bars")
    except Exception as e:
        print(tk, "FAILED:", type(e).__name__, e)

print("\nbar_data keys:", list(bar_data['SPY'].keys()))

In [ ]:
def to_dates(d):
    if 'dates' in d:
        return pd.to_datetime(d['dates'])
    return pd.to_datetime(d['features'].index)

ev_cols = sorted(set().union(*[set(d['features'].columns) for d in event_data.values()]))
bar_cols = sorted(set().union(*[set(d['features'].columns) for d in bar_data.values()]))

ev_parts, bar_parts = [], []
for tk in event_data:
    e = event_data[tk]
    assert len(e['features']) == len(e['labeled_df']) == len(e['labels']), tk
    ev_dates = pd.to_datetime(e['labeled_df']['event_date']).values
    Xe = e['features'].reindex(columns=ev_cols, fill_value=0).fillna(0).reset_index(drop=True)
    ev_parts.append(pd.concat([pd.DataFrame({'ticker': tk, 'date': ev_dates, 'y': e['labels'].values}), Xe], axis=1))

    b = bar_data[tk]
    bd = to_dates(b)
    Xb = b['features'].reindex(columns=bar_cols, fill_value=0).fillna(0).reset_index(drop=True)
    meta = pd.DataFrame({'ticker': tk, 'date': np.asarray(bd), 'y': np.asarray(b['labels'])})
    meta['is_event'] = meta['date'].isin(set(ev_dates))
    bar_parts.append(pd.concat([meta, Xb], axis=1))

EV = pd.concat(ev_parts, ignore_index=True)
BAR = pd.concat(bar_parts, ignore_index=True)
print("event table:", EV.shape, "| bar table:", BAR.shape)
print("share of bars that are events:", round(BAR.is_event.mean(), 3))
print("label mix events:", EV.y.value_counts(normalize=True).round(3).to_dict())
print("label mix bars:  ", BAR.y.value_counts(normalize=True).round(3).to_dict())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

CUT = pd.Timestamp('2022-01-01')
PURGE = pd.Timedelta(days=16)
SEED = 42

def rf(seed=SEED):
    return RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced',
                                  random_state=seed, n_jobs=-1)

def f1m(y, p):
    return f1_score(y, p, average='macro')

def evaluate(table, feats, min_test=20, min_train=30):
    """Returns dict of per-ticker F1 for pooled, per-instrument, dummy.
       For bar tables also returns event-bar-only versions."""
    tr_all = table[table.date < CUT - PURGE]
    pooled = rf().fit(tr_all[feats], tr_all.y)
    out = {k: {} for k in ['pooled', 'own', 'dummy', 'pooled_ev', 'own_ev', 'dummy_ev']}
    for tk, g in table.groupby('ticker'):
        te = g[g.date >= CUT]
        tr = g[g.date < CUT - PURGE]
        if len(te) < min_test or len(tr) < min_train:
            continue
        p_pool = pooled.predict(te[feats])
        p_own = rf().fit(tr[feats], tr.y).predict(te[feats])
        p_dum = DummyClassifier(strategy='stratified', random_state=SEED).fit(tr[feats], tr.y).predict(te[feats])
        out['pooled'][tk] = f1m(te.y, p_pool)
        out['own'][tk] = f1m(te.y, p_own)
        out['dummy'][tk] = f1m(te.y, p_dum)
        if 'is_event' in te.columns:
            m = te.is_event.values
            if m.sum() >= 15:
                out['pooled_ev'][tk] = f1m(te.y[m], p_pool[m])
                out['own_ev'][tk] = f1m(te.y[m], p_own[m])
                out['dummy_ev'][tk] = f1m(te.y[m], p_dum[m])
    return out

res_event = evaluate(EV, ev_cols)
print("events done")
res_bar = evaluate(BAR, bar_cols)
print("bars done")

In [ ]:
def summ(d):
    s = pd.Series(d)
    return dict(n_tickers=len(s), mean_f1=s.mean(), median_f1=s.median())

rows = {
    'Events, pooled':               summ(res_event['pooled']),
    'Events, per-instrument':       summ(res_event['own']),
    'Events, dummy':                summ(res_event['dummy']),
    'Bars, pooled (all test bars)': summ(res_bar['pooled']),
    'Bars, per-instrument (all)':   summ(res_bar['own']),
    'Bars, dummy (all)':            summ(res_bar['dummy']),
    'Bars, pooled (event bars only)':     summ(res_bar['pooled_ev']),
    'Bars, per-instrument (event bars only)': summ(res_bar['own_ev']),
    'Bars, dummy (event bars only)':      summ(res_bar['dummy_ev']),
}
summary = pd.DataFrame(rows).T
pd.options.display.float_format = '{:.3f}'.format
print(summary.to_string())

# per-ticker comparison and head-to-head wins
pt = pd.DataFrame({'ev_pooled': res_event['pooled'], 'ev_own': res_event['own'],
                   'bar_pooled': res_bar['pooled'], 'bar_own': res_bar['own'],
                   'bar_pooled_evonly': res_bar['pooled_ev'], 'ev_dummy': res_event['dummy']})
print("\n", pt.round(3).to_string())
print("\nEvents: pooled beats per-instrument in",
      int((pt.ev_pooled > pt.ev_own).sum()), "of", len(pt), "tickers")
print("Bars:   pooled beats per-instrument in",
      int((pt.bar_pooled > pt.bar_own).sum()), "of", len(pt), "tickers")

ax = summary.loc[[i for i in summary.index if 'dummy' not in i], 'mean_f1'].plot.barh(figsize=(8, 3.5))
ax.axvline(summary.loc['Events, dummy', 'mean_f1'], color='gray', ls='--', label='dummy (events)')
ax.set_xlabel('Mean F1 macro over tickers'); ax.legend(); plt.tight_layout(); plt.show()